# Network 41 — four island assembly repairs

Compare **old current production** with **new corrected isolated geometry** for islands **5, 55, 75 and 138**. All four passed geometry checks. Production networks and RAPID results are unchanged; hydraulic attributes still require regeneration.

Purple **P** labels show the required original parent boundaries. Hover over links/nodes for IDs. Grey = collapsed island footprint; black = remaining land; white = water. Link labels can be enabled in the legend. No `ipywidgets` is required.

The fix uses the complete enclosing unit for computation while filling only the selected island. Island 55 also restores a junction by 3.16 m to P22 and restores exterior L110 exactly.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'RAPID/src/rapid_tools').exists())
sys.path.insert(0, str(ROOT / 'RAPID/src'))
from rapid_tools.island_assembly_review import OUT, ISLANDS, case_data, plot_case
import pandas as pd
import json
from IPython.display import display, HTML

ISLAND = 5  # Choose 5, 55, 75 or 138
BUFFER_M = 150
SHOW_LINK_LABELS = False

In [ ]:
rows = []
for island in ISLANDS:
    r = json.loads((OUT / f'island_41_{island:03d}' / 'review_result.json').read_text())
    rows.append({'island': island, 'geometry accepted': r['geometry_accepted'],
                 'all boundaries present': all(b['present'] for b in r['boundary_checks']),
                 'exterior exact': r['exterior_exact'], 'loops removed': r['cycle_reduction'],
                 'directed acyclic': r['topology']['dag']})
display(pd.DataFrame(rows).set_index('island'))

In [ ]:
fig = plot_case(ISLAND, buffer_m=BUFFER_M, show_link_labels=SHOW_LINK_LABELS)
fig.show(config={'scrollZoom': True, 'displaylogo': False})

In [ ]:
data = case_data(ISLAND, BUFFER_M)
display(pd.DataFrame(data['result']['boundary_checks']))
print('Corrected QGIS geometry:')
print(data['folder'] / 'fixed_links.gpkg')
print(data['folder'] / 'fixed_nodes.gpkg')
print('Old current links:', __import__('rapid_tools.collapse_creation_review', fromlist=['case_row']).case_row('41', data['state']).child_links_path)